In [1]:
# Packages import
import os
current_dir = os.getcwd()
os.chdir('..')
print(f'Moving from {current_dir} to {os.getcwd()}')
import numpy as np
import yaml
import time
from source.utils.masks import *
from source.utils.misc import *
from source.utils.assignment import *
from source.utils.dataset import *
from source.utils.eval import *
from source.core.admm import *

Moving from /Users/mqsirera/RESEARCH/CaP2/sandbox to /Users/mqsirera/RESEARCH/CaP2


In [2]:
# Files to load
config_path = './config/cifar100.yaml' 
partition_path = './config/resnet101-np4.yaml' 

In [3]:
def load_yaml(filepath):
    with open(filepath, 'r') as stream:
        try:
            data = yaml.load(stream, yaml.FullLoader)
        except yaml.YAMLError as exc:
            print(exc)
    return data

In [4]:
# Define config and partition files
configs = load_yaml(config_path)
configs['partition_path'] = partition_path
model = get_model_from_code(configs)

In [5]:
configs = partition_generator(configs, model)

num_partition: {'conv1.weight': 4, 'inputs': 4, 'layer1.0.conv1.weight': 4, 'layer1.0.conv2.weight': 4, 'layer1.1.conv1.weight': 4, 'layer1.1.conv2.weight': 4, 'layer1.2.conv1.weight': 4, 'layer1.2.conv2.weight': 4, 'layer2.0.conv1.weight': 4, 'layer2.0.conv2.weight': 4, 'layer2.0.shortcut.0.weight': 4, 'layer2.1.conv1.weight': 4, 'layer2.1.conv2.weight': 4, 'layer2.2.conv1.weight': 4, 'layer2.2.conv2.weight': 4, 'layer2.3.conv1.weight': 4, 'layer2.3.conv2.weight': 4, 'layer3.0.conv1.weight': 4, 'layer3.0.conv2.weight': 4, 'layer3.0.shortcut.0.weight': 4, 'layer3.1.conv1.weight': 4, 'layer3.1.conv2.weight': 4, 'layer3.10.conv1.weight': 4, 'layer3.10.conv2.weight': 4, 'layer3.11.conv1.weight': 4, 'layer3.11.conv2.weight': 4, 'layer3.12.conv1.weight': 4, 'layer3.12.conv2.weight': 4, 'layer3.13.conv1.weight': 4, 'layer3.13.conv2.weight': 4, 'layer3.14.conv1.weight': 4, 'layer3.14.conv2.weight': 4, 'layer3.15.conv1.weight': 4, 'layer3.15.conv2.weight': 4, 'layer3.16.conv1.weight': 4, 'laye

In [6]:
configs['comm_costs'] = set_communication_cost(model, configs['partition'],)

In [7]:
configs['device'] = 'cpu'

In [8]:
input_var = get_input_from_code(configs)

In [9]:
configs['partition'] = featuremap_summary(model, configs['partition'], input_var)

Inference time per data is 114.695311ms.
conv1.weight 1024
layer1.0.conv1.weight 1024
layer1.0.conv2.weight 1024
layer1.1.conv1.weight 1024
layer1.1.conv2.weight 1024
layer1.2.conv1.weight 1024
layer1.2.conv2.weight 1024
layer2.0.conv1.weight 256
layer2.0.conv2.weight 256
layer2.0.shortcut.0.weight 256
layer2.1.conv1.weight 256
layer2.1.conv2.weight 256
layer2.2.conv1.weight 256
layer2.2.conv2.weight 256
layer2.3.conv1.weight 256
layer2.3.conv2.weight 256
layer3.0.conv1.weight 64
layer3.0.conv2.weight 64
layer3.0.shortcut.0.weight 64
layer3.1.conv1.weight 64
layer3.1.conv2.weight 64
layer3.2.conv1.weight 64
layer3.2.conv2.weight 64
layer3.3.conv1.weight 64
layer3.3.conv2.weight 64
layer3.4.conv1.weight 64
layer3.4.conv2.weight 64
layer3.5.conv1.weight 64
layer3.5.conv2.weight 64
layer3.6.conv1.weight 64
layer3.6.conv2.weight 64
layer3.7.conv1.weight 64
layer3.7.conv2.weight 64
layer3.8.conv1.weight 64
layer3.8.conv2.weight 64
layer3.9.conv1.weight 64
layer3.9.conv2.weight 64
layer3.10.

In [10]:
train_loader, test_loader = get_dataset_from_code(configs['data_code'], configs['batch_size'])

Files already downloaded and verified
Files already downloaded and verified


## Training

In [11]:
evalHelper = EvalHelper(configs['data_code'])

def test_model(model, criterion, cepoch=0):
        acc = evalHelper.get_accuracy(model, test_loader, criterion, cepoch)
        return acc

In [14]:
def standard_train(configs, cepoch, model, data_loader, criterion, optimizer, scheduler, ADMM=None, masks=None, comm=False):

    batch_acc    = AverageMeter()
    batch_loss   = AverageMeter()
    batch_comm   = AverageMeter()
    evalHelper   = EvalHelper(configs['data_code'])
    
    if comm:
        partition = configs['partition']
    
    if ADMM is not None: 
        admm_initialization(configs, ADMM=ADMM, model=model)
        
    start_time = time.time()
    n_data = configs['batch_size'] * len(data_loader)
    pbar = tqdm(enumerate(data_loader), total=n_data/configs['batch_size'], ncols=150)
    
    for batch_idx, batch in pbar:
           
        data   = ()
        for piece in batch[:-1]:
            data += (piece.float().to(configs['device']),)
        target = batch[-1].to(configs['device'])
        total_loss = 0
        comm_loss = 0
        comp_loss = 0

        data = (torch.cat(data, dim=1),)
        
        optimizer.zero_grad()
        
        if configs['mix_up']:
            data, target_a, target_b, lam = mixup_data(*data, y=target, alpha=configs['alpha'])
        # print('data:', data)
        
        # print(len(data))
        # print(data[0].shape)
        output = model(*data)
        # print('output:', output)
        
        if configs['mix_up']:
            loss = mixup_criterion(criterion, output, target_a, target_b, lam, configs['smooth'])
        else:
            loss = criterion(output, target, smooth=configs['smooth'])
            # loss = criterion(output, target.unsqueeze(1).float())
        # print('xentropy_loss:', loss)
        total_loss += (loss * configs['xentropy_weight'])
        # print('total_loss:', total_loss)
        
        if ADMM is not None:
            z_u_update(configs, ADMM, model, cepoch, batch_idx)  # update Z and U variables
            prev_loss, admm_loss, total_loss = append_admm_loss(ADMM, model, total_loss)  # append admm losses
            
        if comm:
            for (name, W) in model.named_parameters():
                if name in ADMM.prune_ratios:
                    comm_cost = torch.abs(W) * configs['comm_costs'][name]
                    '''
                    v1: abs(W)*comm_cost
                    '''
                    comm_cost = comm_cost.view(comm_cost.size(0), -1).sum()
                    if configs['comm_outsize']:
                        comm_loss += comm_cost*partition[name]['outsize']
                    else:
                        comm_loss += comm_cost
                    
                    '''
                    #v2: further constraint on max(abs(W)*comm_cost)
                    '''
                    '''
                    comm_cost = comm_cost.reshape(W.shape[0], W.shape[1], -1).sum(-1)
                    for i in range(partition[name]['num']):
                        for j in range(partition[name]['num']):
                            if i==j: continue
                            cost_interp = comm_cost[partition[name]['filter_id'][i][:,None],
                                                    partition[name]['channel_id'][j]].sum()
                            #comm_loss += cost_interp*partition[name]['outsize']) #p_{count}
                            comm_loss = max(comm_loss,cost_interp*partition[name]['outsize']) # p_{max}
                            
                            #comp_loss = max(comp_loss, cost_interp*partition[name]['outsize'])
                    '''
                    '''
                    computation cost:
                    for i in range(partition[name]['num']):
                        comp_loss = max(comp_loss, torch.abs(W).view(W.size(0), -1)[partition[name]['filter_id'][i],:].sum())
                    '''
            total_loss += configs['lambda_comm'] * comm_loss + configs['lambda_comp'] * comp_loss
            # print('total_loss:', total_loss)
        
        total_loss.backward() # Back Propagation
        
        # For masked training
        if masks is not None:
            with torch.no_grad():
                for name, W in (model.named_parameters()):
                    if name in masks and W.grad is not None:
                        W.grad *= masks[name]
                        
        optimizer.step()
        
        # adjust learning rate
        if ADMM is not None:
            admm_adjust_learning_rate(optimizer, cepoch, configs)
        else:
            scheduler.step()

        # Reassign neurons to machines
        if configs['reassign'] and (batch_idx+1) % configs['reassign_freq'] == 0:
            print('Updating assignment')
            update_time = time.time()
            update_assignments(model, configs)
            configs['comm_costs'] = set_communication_cost(model, configs['partition'])
            print(f'Assignment ellapsed {time.time()-update_time} ms')


        acc1 = evalHelper.call(output, target)
        batch_loss.update(loss.item(), target.size(0))
        batch_comm.update(comm_loss.item() if comm_loss else comm_loss, target.size(0))
        batch_acc.update(acc1[0].item(), target.size(0))

        
        # # # preparation log information and print progress # # #
        msg = 'Train Epoch: {cepoch} [ {cidx:5d}/{tolidx:5d} ({perc:2d}%)] Loss:{loss:.4f} CommLoss:{commloss:.4f} Acc:{acc:.4f}'.format(
                        cepoch = cepoch,  
                        cidx = (batch_idx+1)*configs['batch_size'], 
                        tolidx = n_data,
                        perc = int(100. * (batch_idx+1)*configs['batch_size']/n_data), 
                        loss = batch_loss.avg,
                        commloss = batch_comm.avg,
                        acc  = batch_acc.avg,
                    )

        pbar.set_description(msg)
    #print('Training time per epoch is {:.2f}s.'.format(time.time()-start_time))

In [17]:
configs['partition']

{'bn_partition': [4, 4, 4, 4, 4, 4, 4, 4, 4],
 'conv1.weight': {'num': 4,
  'filter_id': [array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15]),
   array([16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]),
   array([32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47]),
   array([48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63]),
   array([], dtype=int64)],
  'channel_id': [array([], dtype=int64),
   array([0]),
   array([1]),
   array([2]),
   array([], dtype=int64)],
  'maps': [[0, 1, 1, 1], [1, 0, 1, 1], [1, 1, 0, 1], [1, 1, 1, 0]],
  'outsize': 1024},
 'layer1.0.conv1.weight': {'num': 4,
  'filter_id': [array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15]),
   array([16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]),
   array([32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47]),
   array([48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63]),
   array([], dtype=int64)

In [15]:
nepoch = configs['epochs']
criterion, optimizer, scheduler = set_optimizer(configs, model, train_loader, \
                                    configs['optimizer'], configs['learning_rate'], nepoch)

# Initializing ADMM; if not admm, do hard pruning only
admm = ADMM(configs, model, rho=configs['rho']) if configs['admm'] else None

# prune
for cepoch in range(0, nepoch+1):
    if cepoch>0:
        print('Learning rate: {:.4f}'.format(get_lr(optimizer)))
        standard_train(configs, cepoch, model, train_loader, 
                    criterion, optimizer, scheduler, ADMM=admm, comm=True)
    acc = test_model(model, criterion, cepoch)

{'conv1.weight': 0.5, 'layer1.0.conv1.weight': 0.5, 'layer1.0.conv2.weight': 0.5, 'layer1.1.conv1.weight': 0.5, 'layer1.1.conv2.weight': 0.5, 'layer1.2.conv1.weight': 0.5, 'layer1.2.conv2.weight': 0.5, 'layer2.0.conv1.weight': 0.5, 'layer2.0.conv2.weight': 0.5, 'layer2.0.shortcut.0.weight': 0.5, 'layer2.1.conv1.weight': 0.5, 'layer2.1.conv2.weight': 0.5, 'layer2.2.conv1.weight': 0.5, 'layer2.2.conv2.weight': 0.5, 'layer2.3.conv1.weight': 0.5, 'layer2.3.conv2.weight': 0.5, 'layer3.0.conv1.weight': 0.5, 'layer3.0.conv2.weight': 0.5, 'layer3.0.shortcut.0.weight': 0.5, 'layer3.1.conv1.weight': 0.5, 'layer3.1.conv2.weight': 0.5, 'layer3.2.conv1.weight': 0.5, 'layer3.2.conv2.weight': 0.5, 'layer3.3.conv1.weight': 0.5, 'layer3.3.conv2.weight': 0.5, 'layer3.4.conv1.weight': 0.5, 'layer3.4.conv2.weight': 0.5, 'layer3.5.conv1.weight': 0.5, 'layer3.5.conv2.weight': 0.5, 'layer3.6.conv1.weight': 0.5, 'layer3.6.conv2.weight': 0.5, 'layer3.7.conv1.weight': 0.5, 'layer3.7.conv2.weight': 0.5, 'layer3.

  0%|                                                                                                                       | 0/391.0 [00:00<?, ?it/s]

Updating assignment


Train Epoch: 1 [   128/50048 ( 0%)] Loss:4.6049 CommLoss:2367851.8463 Acc:2.3438:   0%|                           | 1/391.0 [00:16<1:45:24, 16.22s/it]

Assignment ellapsed 0.339461088180542 ms
Updating assignment


...

Assignment ellapsed 0.20534801483154297 ms


Train Epoch: 1 [  2560/50048 ( 5%)] Loss:4.6089 CommLoss:1481307.8132 Acc:0.9375:   5%|█▎                        | 20/391.0 [04:41<1:28:14, 14.27s/it]

Updating assignment
Assignment ellapsed 0.19078683853149414 ms


Train Epoch: 1 [  2560/50048 ( 5%)] Loss:4.6089 CommLoss:1481307.8132 Acc:0.9375:   5%|█▎                        | 20/391.0 [04:56<1:31:33, 14.81s/it]


KeyboardInterrupt: 

In [15]:
configs['partition']

{'bn_partition': [4, 4, 4, 4, 4, 4, 4, 4, 4],
 'conv1.weight': {'num': 4,
  'filter_id': [array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15]),
   array([16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]),
   array([32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47]),
   array([48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63]),
   array([], dtype=int64)],
  'channel_id': [array([], dtype=int64),
   array([0]),
   array([1]),
   array([2]),
   array([], dtype=int64)],
  'maps': [[0, 1, 1, 1], [1, 0, 1, 1], [1, 1, 0, 1], [1, 1, 1, 0]],
  'outsize': 1024},
 'layer1.0.conv1.weight': {'num': 4,
  'filter_id': [array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15]),
   array([16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]),
   array([32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47]),
   array([48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63]),
   array([], dtype=int64)

## Tests

In [7]:
# Partition information
partition = {
    'conv1.weight': {
        'num': 3,  # 3 partitions
        'filter_id': [np.array([0, 1]), np.array([2, 3]), np.array([4, 5])],  # Filters per partition
        'channel_id': [np.array([0, 1]), np.array([2]), np.array([3])],  # Input channels per partition
        'maps': [[0, 1, 2], [1, 0, 1], [2, 1, 0]]  # Communication cost maps
    }
}

# Weights for conv1.weight layer (4D: Conv2D)
weights = torch.tensor([
    [[[1, 0], [0, 0]], [[0, 0], [0, 0]], [[0, 0], [0, 0]], [[0, 0], [0, 0]]],  # Filter 0
    [[[0, 0], [0, 0]], [[1, 0], [0, 0]], [[0, 0], [1, 0]], [[0, 0], [0, 1]]],  # Filter 1
    [[[1, 1], [1, 0]], [[0, 0], [0, 0]], [[1, 0], [0, 1]], [[0, 0], [0, 1]]],  # Filter 2
    [[[0, 0], [1, 0]], [[1, 0], [0, 0]], [[0, 1], [0, 0]], [[0, 0], [0, 0]]],  # Filter 3
    [[[0, 1], [0, 0]], [[0, 0], [1, 1]], [[1, 0], [0, 0]], [[0, 0], [0, 0]]],  # Filter 4
    [[[1, 0], [0, 0]], [[0, 1], [1, 1]], [[0, 0], [0, 0]], [[0, 0], [1, 0]]]   # Filter 5
])



In [9]:
cost_matrix = compute_cost_matrix('conv1.weight', weights, partition)
print(cost_matrix)


[[0. 1. 2.]
 [3. 2. 3.]
 [3. 2. 3.]
 [1. 2. 5.]
 [1. 2. 5.]
 [2. 3. 4.]]


In [15]:
current_time = time.time()
num_layers = 0
print(f'Start time: {current_time} s')
for name, W in model.named_parameters():
    if name in configs['partition']:
        C = compute_cost_matrix(name, W, configs['partition'])
        num_layers += 1
        print(f'Layer {name} processed in: {time.time() - current_time} s')
        print(f'C shape: {len(C)}, {len(C[0])}')
        current_time = time.time()
        
print(f'Num. layers: {num_layers}') 

Start time: 1732648294.427691 s
Layer conv1.weight processed in: 0.020582914352416992 s
C shape: 64, 4
Layer layer1.0.conv1.weight processed in: 0.01493525505065918 s
C shape: 64, 4
Layer layer1.0.conv2.weight processed in: 0.011151790618896484 s
C shape: 64, 4
Layer layer1.1.conv1.weight processed in: 0.009544134140014648 s
C shape: 64, 4
Layer layer1.1.conv2.weight processed in: 0.008366107940673828 s
C shape: 64, 4
Layer layer1.2.conv1.weight processed in: 0.0076978206634521484 s
C shape: 64, 4
Layer layer1.2.conv2.weight processed in: 0.00710606575012207 s
C shape: 64, 4
Layer layer2.0.conv1.weight processed in: 0.01325225830078125 s
C shape: 128, 4
Layer layer2.0.conv2.weight processed in: 0.014045000076293945 s
C shape: 128, 4
Layer layer2.0.shortcut.0.weight processed in: 0.011737823486328125 s
C shape: 128, 4
Layer layer2.1.conv1.weight processed in: 0.013861894607543945 s
C shape: 128, 4
Layer layer2.1.conv2.weight processed in: 0.014148950576782227 s
C shape: 128, 4
Layer lay

In [26]:
C

array([[384., 384., 384., 384.],
       [384., 384., 384., 384.],
       [384., 384., 384., 384.],
       ...,
       [384., 384., 384., 384.],
       [384., 384., 384., 384.],
       [384., 384., 384., 384.]])

In [ ]:
def matrix_to_partition(P, original_partition, previous_partition):
    """
    Translates an assignment matrix P (neurons by machines) back into a partition dictionary.

    Args:
        P (np.ndarray): A binary matrix of shape (num_neurons, num_partitions), where:
                        - P[i, j] = 1 if output neuron `i` is executed on machine `j`, 0 otherwise.
        original_partition (dict): The original partition dictionary, used to retrieve:
                                   - maps: Communication cost between partitions
        previous_partition (dict): The previous layer partition dictionary, used to retrieve:
                                   - channel_id: Input channels per partition

    Returns:
        dict: A new partition dictionary reconstructed based on P.
    """
    # Ensure P is a NumPy array
    P = np.array(P)

    # Validate input dimensions
    num_neurons, num_partitions = P.shape
    if 'num' not in original_partition or original_partition['num'] != num_partitions:
        raise ValueError("Mismatch between P's number of partitions and the original partition dictionary.")

    # Reconstruct the partition dictionary
    new_partition = {
        'num': num_partitions,
        'filter_id': [],  # Output neurons (filters) per partition
        'channel_id': previous_partition['filter_id'],  # Retain original input channels
        'maps': original_partition['maps'],  # Retain original communication cost map
    }

    # Populate filter_id for each partition
    for j in range(num_partitions):
        new_partition['filter_id'].append(np.where(P[:, j] == 1)[0])

    return new_partition


In [ ]:
P = np.array([
    [1, 0, 0],  # Neuron 0 -> Partition 0
    [1, 0, 0],  # Neuron 1 -> Partition 0
    [0, 1, 0],  # Neuron 2 -> Partition 1
    [0, 1, 0],  # Neuron 3 -> Partition 1
    [0, 0, 1],  # Neuron 4 -> Partition 2
    [0, 0, 1],  # Neuron 5 -> Partition 2
])

original_partition = {
    'num': 3,
    'filter_id': [np.array([0, 1]), np.array([2, 3]), np.array([4, 5])],
    'channel_id': [np.array([0, 1]), np.array([2]), np.array([3])],
    'maps': [[0, 1, 2], [1, 0, 1], [2, 1, 0]],
}

In [ ]:
new_partition = matrix_to_partition(P, original_partition)
print(new_partition)